# IndicEAR: train the released model on Colab

The recipe of the released model (`docs/training-recipe.md`), in two stages on one GPU runtime:

1. **Pilot** (`scripts/pilot.sh`): Hindi, Tamil, Kannada and English from the frozen encoder; about 4.3 A100-hours (less on a G4).
2. **Full warm-started run** (`scripts/full_warm.sh`): all 12 languages, warm-started from the pilot; about 6 hours on an RTX PRO 6000 (Colab G4, about 55 units for training).

Every long step is a bash script started with `scripts/launch.sh` under `nohup`, so it survives notebook disconnects; every step resumes when rerun. Training audio streams from the Hugging Face Hub, so a Hugging Face token (Colab secret `HF_TOKEN`, with the gated datasets' terms accepted) is needed.

**Runtime:** Colab G4 (RTX PRO 6000, 96 GB) recommended, or an A100 (80 GB). Batch budgets are sized from free VRAM, so no edits are needed.

In [ ]:
# @title 1. GPU, CPU, RAM and disk
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!nproc; free -g | head -2; df -h /content | tail -1

In [ ]:
# @title 2. Code, token and environment (free-threaded Python 3.14t, exact pins)
!git clone -q https://github.com/rahulsharmavishwakarma/indicear /content/indicear-code
%env CODE=/content/indicear-code
%env VENV=/content/venv314t
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!bash $CODE/scripts/setup_env.sh

In [ ]:
# @title 3. Stage 1: the pilot (background: setup, prep, train, eval, report)
%env ROOT=/content/pilot
!bash $CODE/scripts/pilot.sh

In [ ]:
# @title Progress (rerun any time)
!bash $CODE/scripts/status.sh

In [ ]:
# @title 4. Warm-start source: the pilot's final v4+lid weights and tokenizer
!mkdir -p /content/full/init/v4+lid
!cd /content/pilot/runs/v4+lid && cp heads.safetensors adapters.safetensors variant.json lid.json result.json /content/full/init/v4+lid/
!cp -r /content/pilot/tok /content/full/init/v4+lid/tok && ls /content/full/init/v4+lid

In [ ]:
# @title 5. Stage 2: the full warm-started run (background: setup, prep, train, eval, report)
# The released run added --zero-streams at step 1,079 and --stream-scale at step 3,427 (see the recipe's
# "Training at scale"); here they apply from the start.
%env ROOT=/content/full
%env INIT_FROM=/content/full/init/v4+lid
%env TRAIN_EXTRA=--new-script-bias -2 --cache-gb 100 --readahead 64 --zero-streams kathbath/ml,kathbath/or,kathbath/ur,kathbath/gu --stream-scale indicvoices/hi=2.24,vaani/hi=2.24
!bash $CODE/scripts/full_warm.sh

In [ ]:
# @title Progress (rerun any time): step, loss, throughput, GPU use
!bash $CODE/scripts/status.sh
!tail -n 3 /content/full/runs/v4+lid/train.csv

In [ ]:
# @title 6. Package the model (with the audio tower) and try it
!cd $CODE && $VENV/bin/python -m indicear.export --run /content/full/runs/v4+lid --tok /content/full/tok --out /content/indicear-model --with-encoder
import json
clip = json.loads(open('/content/full/manifests/eval_kathbath_clean_test_unknown_hi.jsonl').readline())
print('reference:', clip['text'])
!cd $CODE && $VENV/bin/python -m indicear.transcribe --model /content/indicear-model {clip['audio']}

In [ ]:
# @title 7. Results; copy what you want to keep, then release the runtime
!cat /content/full/results/eval_summary.md | head -60
print('Copy /content/indicear-model and /content/full/results, then: Runtime > Disconnect and delete runtime.')